# Results Analysis — Best Model vs Baseline

Inspect **held-out test** examples from `training_pipeline.ipynb`:

1. **Wins** — best model correct where baseline (`vixid_vanilla`) is wrong
2. **Baseline wins** — baseline correct where best model is wrong
3. **Shared failures** — both models wrong

Uses cached predictions under `runs/_significance_preds/` (same order as `test_df`) and reconstructs the identical stratified split (`SEED=42`).

Best model defaults to `models/best_ev_ice_classifier/metadata.json` → currently **`yolo26s_cls`**.

## 1) Imports & config

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split


In [ ]:
TRAINER_ROOT = Path.cwd().resolve()
DATASET_ROOT = TRAINER_ROOT / "dataset"
ANNOTATIONS_CSV = DATASET_ROOT / "annotations.csv"
RUNS_DIR = TRAINER_ROOT / "runs"
PRED_DIR = RUNS_DIR / "_significance_preds"
EXPORT_META = TRAINER_ROOT / "models" / "best_ev_ice_classifier" / "metadata.json"

VALID_LABELS = ("ev", "ice")
CLASS_TO_IDX = {"ev": 0, "ice": 1}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}

SEED = 42
VAL_SIZE = 0.20
TEST_SIZE = 0.15

BASELINE_NAME = "vixid_vanilla"

BEST_MODEL_NAME: str | None = None

MAX_GALLERY_IMAGES = 48
N_COLS = 4
FIGSIZE_PER_IMAGE = (3.2, 3.6)

print(f"Trainer root : {TRAINER_ROOT}")
print(f"Predictions  : {PRED_DIR}")
print(f"Baseline     : {BASELINE_NAME}")


## 2) Rebuild the same test split

In [ ]:
def load_labeled_dataset(csv_path: Path, dataset_root: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path).copy()
    required = {"relative_path", "label"}
    missing_cols = required - set(df.columns)
    if missing_cols:
        raise ValueError(f"annotations.csv missing columns: {sorted(missing_cols)}")

    df["label"] = df["label"].astype(str).str.strip().str.lower()
    df = df[df["label"].isin(VALID_LABELS)].copy()
    df["image_path"] = df["relative_path"].map(lambda rel: dataset_root / rel)
    df["exists"] = df["image_path"].map(lambda p: Path(p).exists())

    missing = df[~df["exists"]]
    if len(missing):
        print(f"Dropping {len(missing)} rows with missing image files")

    df = df[df["exists"]].reset_index(drop=True)
    df["label_idx"] = df["label"].map(CLASS_TO_IDX).astype(int)
    return df

def split_dataframe(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    train_df, test_df = train_test_split(
        df,
        test_size=TEST_SIZE,
        random_state=SEED,
        stratify=df["label_idx"],
    )
    val_ratio = VAL_SIZE / (1.0 - TEST_SIZE)
    train_df, val_df = train_test_split(
        train_df,
        test_size=val_ratio,
        random_state=SEED,
        stratify=train_df["label_idx"],
    )
    return (
        train_df.reset_index(drop=True),
        val_df.reset_index(drop=True),
        test_df.reset_index(drop=True),
    )

labeled_df = load_labeled_dataset(ANNOTATIONS_CSV, DATASET_ROOT)
_, _, test_df = split_dataframe(labeled_df)

print(f"Usable labeled images: {len(labeled_df)}")
print(f"Test set size        : {len(test_df)}")
print(test_df["label"].value_counts())
test_df.head(3)


## 3) Load cached predictions & build comparison table

In [ ]:
def resolve_best_model_name(override: str | None = None) -> str:
    if override:
        return override
    if EXPORT_META.exists():
        meta = json.loads(EXPORT_META.read_text())
        name = meta.get("exported_experiment") or meta.get("best_overall_experiment")
        if name:
            return str(name)
    raise FileNotFoundError(
        f"Could not resolve best model. Set BEST_MODEL_NAME or create {EXPORT_META}"
    )

def load_pred_json(name: str) -> dict:
    path = PRED_DIR / f"{name}_test_preds.json"
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {path}. Re-run the significance section in training_pipeline.ipynb first."
        )
    return json.loads(path.read_text())

BEST_MODEL_NAME = resolve_best_model_name(BEST_MODEL_NAME)
baseline_preds = load_pred_json(BASELINE_NAME)
best_preds = load_pred_json(BEST_MODEL_NAME)

y_true = np.asarray(baseline_preds["y_true"], dtype=int)
y_pred_base = np.asarray(baseline_preds["y_pred"], dtype=int)
y_pred_best = np.asarray(best_preds["y_pred"], dtype=int)
y_prob_base = np.asarray(baseline_preds["y_prob_ev"], dtype=float)
y_prob_best = np.asarray(best_preds["y_prob_ev"], dtype=float)

if len(y_true) != len(test_df):
    raise RuntimeError(f"Length mismatch: preds={len(y_true)} vs test_df={len(test_df)}")
if not np.array_equal(y_true, test_df["label_idx"].to_numpy()):
    raise RuntimeError("Prediction y_true order does not match reconstructed test_df")
if not np.array_equal(y_true, np.asarray(best_preds["y_true"], dtype=int)):
    raise RuntimeError("Baseline and best-model y_true arrays disagree")

analysis_df = test_df.copy()
analysis_df["y_true"] = y_true
analysis_df["gt_label"] = analysis_df["y_true"].map(IDX_TO_CLASS)
analysis_df["pred_baseline"] = y_pred_base
analysis_df["pred_best"] = y_pred_best
analysis_df["pred_baseline_label"] = analysis_df["pred_baseline"].map(IDX_TO_CLASS)
analysis_df["pred_best_label"] = analysis_df["pred_best"].map(IDX_TO_CLASS)
analysis_df["prob_ev_baseline"] = y_prob_base
analysis_df["prob_ev_best"] = y_prob_best
analysis_df["baseline_correct"] = analysis_df["pred_baseline"] == analysis_df["y_true"]
analysis_df["best_correct"] = analysis_df["pred_best"] == analysis_df["y_true"]

wins_df = analysis_df[~analysis_df["baseline_correct"] & analysis_df["best_correct"]].copy()
both_fail_df = analysis_df[~analysis_df["baseline_correct"] & ~analysis_df["best_correct"]].copy()
baseline_only_df = analysis_df[analysis_df["baseline_correct"] & ~analysis_df["best_correct"]].copy()

print(f"Best model   : {BEST_MODEL_NAME}")
print(f"Baseline     : {BASELINE_NAME}")
print(f"Test n       : {len(analysis_df)}")
print(f"Baseline acc : {baseline_preds['accuracy']:.4f} | F1_ev={baseline_preds['f1_ev']:.4f}")
print(f"Best acc     : {best_preds['accuracy']:.4f} | F1_ev={best_preds['f1_ev']:.4f}")
print()
print(f"Wins (best ✓, baseline ✗)     : {len(wins_df)}")
print(f"Shared failures (both ✗)      : {len(both_fail_df)}")
print(f"Baseline-only correct (info)  : {len(baseline_only_df)}")


In [ ]:
summary = pd.DataFrame(
    {
        "bucket": [
            "both_correct",
            "best_wins",
            "baseline_only_correct",
            "both_fail",
        ],
        "count": [
            int((analysis_df["baseline_correct"] & analysis_df["best_correct"]).sum()),
            len(wins_df),
            len(baseline_only_df),
            len(both_fail_df),
        ],
    }
)
summary["pct_of_test"] = (100 * summary["count"] / len(analysis_df)).round(2)
display(summary)

print("\nWins by ground-truth class:")
display(wins_df["gt_label"].value_counts().rename("n").to_frame())

print("\nBaseline-only correct by ground-truth class:")
display(baseline_only_df["gt_label"].value_counts().rename("n").to_frame())

print("\nShared failures by ground-truth class:")
display(both_fail_df["gt_label"].value_counts().rename("n").to_frame())


## 4) Image gallery helpers

Each tile shows the crop, **GT**, baseline prediction + EV prob, and best-model prediction + EV prob.

In [ ]:
def _read_rgb(path: Path) -> np.ndarray | None:
    bgr = cv2.imread(str(path))
    if bgr is None:
        return None
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

def _title_color(gt: str, pred: str) -> str:
    return "#2a9d8f" if pred == gt else "#e63946"

def show_prediction_gallery(
    frame: pd.DataFrame,
    *,
    title: str,
    max_images: int | None = MAX_GALLERY_IMAGES,
    n_cols: int = N_COLS,
    sort_by: str | None = "prob_ev_best",
    ascending: bool = False,
) -> None:
    if frame.empty:
        print(f"{title}: no examples.")
        return

    view = frame.copy()
    if sort_by and sort_by in view.columns:
        view = view.sort_values(sort_by, ascending=ascending)

    if max_images is not None:
        view = view.head(int(max_images))

    n = len(view)
    n_cols = max(1, int(n_cols))
    n_rows = int(np.ceil(n / n_cols))
    fig_w = FIGSIZE_PER_IMAGE[0] * n_cols
    fig_h = FIGSIZE_PER_IMAGE[1] * n_rows
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(fig_w, fig_h))
    axes_arr = np.atleast_1d(axes).ravel()

    for ax, (_, row) in zip(axes_arr, view.iterrows()):
        img = _read_rgb(Path(row["image_path"]))
        if img is None:
            ax.text(0.5, 0.5, "missing image", ha="center", va="center")
            ax.axis("off")
            continue

        ax.imshow(img)
        ax.axis("off")

        gt = str(row["gt_label"])
        base_pred = str(row["pred_baseline_label"])
        best_pred = str(row["pred_best_label"])
        rel = str(row["relative_path"])

        caption = (
            f"{rel}\n"
            f"GT: {gt.upper()}\n"
            f"baseline: {base_pred.upper()}  (p_ev={row['prob_ev_baseline']:.2f})\n"
            f"best: {best_pred.upper()}  (p_ev={row['prob_ev_best']:.2f})"
        )
        ax.set_title(
            caption,
            fontsize=8,
            color=_title_color(gt, best_pred),
            loc="left",
            fontfamily="monospace",
        )

        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(3)
            spine.set_color(_title_color(gt, best_pred))

    for ax in axes_arr[n:]:
        ax.axis("off")

    fig.suptitle(f"{title}  (showing {n} / {len(frame)})", fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

def show_compact_table(frame: pd.DataFrame, *,
                       cols: list[str] | None = None,
                       max_rows: int = 40) -> None:
    cols = cols or [
        "relative_path",
        "gt_label",
        "pred_baseline_label",
        "prob_ev_baseline",
        "pred_best_label",
        "prob_ev_best",
    ]
    view = frame[cols].copy()
    view["prob_ev_baseline"] = view["prob_ev_baseline"].map(lambda x: f"{x:.3f}")
    view["prob_ev_best"] = view["prob_ev_best"].map(lambda x: f"{x:.3f}")
    display(view.head(max_rows))
    if len(frame) > max_rows:
        print(f"... {len(frame) - max_rows} more rows not shown")


## 5) Wins — best correct, baseline wrong

Green titles = best model matches GT. Sorted by best-model EV probability (highest first).

In [ ]:
print(f"Total wins: {len(wins_df)}")
show_compact_table(wins_df.sort_values("prob_ev_best", ascending=False))
show_prediction_gallery(
    wins_df,
    title=f"Wins: {BEST_MODEL_NAME} ✓  |  {BASELINE_NAME} ✗",
    sort_by="prob_ev_best",
    ascending=False,
)


In [ ]:
for gt_cls in ("ev", "ice"):
    subset = wins_df[wins_df["gt_label"] == gt_cls]
    show_prediction_gallery(
        subset,
        title=f"Wins on GT={gt_cls.upper()}  ({len(subset)})",
        sort_by="prob_ev_best",
        ascending=(gt_cls == "ice"),
    )


## 6) Shared failures — best model also wrong

Red titles = best model disagrees with GT. Useful for finding hard / ambiguous crops.

In [ ]:
print(f"Total shared failures: {len(both_fail_df)}")
show_compact_table(both_fail_df.sort_values("prob_ev_best", ascending=False))
show_prediction_gallery(
    both_fail_df,
    title=f"Shared failures: {BEST_MODEL_NAME} ✗  and  {BASELINE_NAME} ✗",
    sort_by="prob_ev_best",
    ascending=False,
)


In [ ]:
for gt_cls in ("ev", "ice"):
    subset = both_fail_df[both_fail_df["gt_label"] == gt_cls]
    show_prediction_gallery(
        subset,
        title=f"Shared failures on GT={gt_cls.upper()}  ({len(subset)})",
        sort_by="prob_ev_best",
        ascending=False,
    )


## 7) Baseline wins — baseline correct, best model wrong

Cases where `vixid_vanilla` still beats the best model. Red titles = best model disagrees with GT.


In [ ]:
print(f"Total baseline wins: {len(baseline_only_df)}")
show_compact_table(baseline_only_df.sort_values("prob_ev_best", ascending=False))
show_prediction_gallery(
    baseline_only_df,
    title=f"Baseline wins: {BASELINE_NAME} ✓  |  {BEST_MODEL_NAME} ✗",
    sort_by="prob_ev_best",
    ascending=False,
)


In [ ]:
for gt_cls in ("ev", "ice"):
    subset = baseline_only_df[baseline_only_df["gt_label"] == gt_cls]
    show_prediction_gallery(
        subset,
        title=f"Baseline wins on GT={gt_cls.upper()}  ({len(subset)})",
        sort_by="prob_ev_best",
        ascending=False,
    )


## 8) Export filtered CSVs (optional)

Write the win / failure tables for offline review.


In [ ]:
out_dir = RUNS_DIR / "results_analysis"
out_dir.mkdir(parents=True, exist_ok=True)

export_cols = [
    "relative_path",
    "image_path",
    "gt_label",
    "pred_baseline_label",
    "prob_ev_baseline",
    "pred_best_label",
    "prob_ev_best",
    "baseline_correct",
    "best_correct",
]

wins_path = out_dir / f"wins_{BEST_MODEL_NAME}_over_{BASELINE_NAME}.csv"
fail_path = out_dir / f"shared_failures_{BEST_MODEL_NAME}_and_{BASELINE_NAME}.csv"
full_path = out_dir / f"test_comparison_{BEST_MODEL_NAME}_vs_{BASELINE_NAME}.csv"

wins_df[export_cols].to_csv(wins_path, index=False)
both_fail_df[export_cols].to_csv(fail_path, index=False)
analysis_df[export_cols].to_csv(full_path, index=False)

print(f"Saved wins            → {wins_path}")
print(f"Saved shared failures → {fail_path}")
print(f"Saved full comparison → {full_path}")
